In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import os
import pickle

from torkin import TorKin
from config import Config

from scipy.spatial.transform import Rotation

In [2]:
cfg = Config()
kin = TorKin()

No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/larm60.txt
bodypart structure for larm is constructed.
No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/rarm60.txt
bodypart structure for rarm is constructed.
bodypart structure for head is constructed.


In [3]:
root_dir = "/home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/"
ds_name = "trajs:72_blocks:3_triangle_v_scarce"
ds_type = "interp_0.85"

In [4]:
data_path = os.path.join(root_dir, f'data/torobo/{ds_name}/train_{ds_type}.npy')
print(data_path)

/home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/data/torobo/trajs:72_blocks:3_triangle_v_scarce/train_interp_0.85.npy


In [5]:
with open(data_path, 'rb') as f:
    data_np = np.load(f)

# data_np = np.load(data_path)
print(data_np.shape)

(61, 299, 35)


# Sample Inspection

In [6]:
print(data_np[0, 11, :])

[ 1.10000000e+01  1.31474233e+00  1.15303148e+00  1.13525614e+00
  1.44941022e+00 -6.38663018e-01  9.38644129e-01  5.91524606e-01
 -1.78511221e-02 -3.92443068e-03 -8.91308725e-03  7.77143212e-03
 -3.90806527e-03 -1.89011501e-02  1.39139268e-03  4.43117333e-01
 -2.42614808e-01  8.65000000e-01  4.74497475e-01 -1.25502525e-01
  8.65000000e-01  3.57385192e-01 -1.56882667e-01  8.65000000e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -1.77693587e-02 -3.83580150e-03 -8.91295094e-03  7.48425979e-03
 -3.91306625e-03 -1.89848328e-02  1.31651868e-03]


In [7]:
joint_dim = 7
print(data_np[0, 11, 1:1+joint_dim]) #Joints

[ 1.31474233  1.15303148  1.13525614  1.44941022 -0.63866302  0.93864413
  0.59152461]


In [8]:
print(data_np[0, 11, 8:8+joint_dim]) #Velocs

[-0.01785112 -0.00392443 -0.00891309  0.00777143 -0.00390807 -0.01890115
  0.00139139]


In [9]:
print(data_np[0, 11, 15:15+13]) #Target

[ 0.44311733 -0.24261481  0.865       0.47449747 -0.12550253  0.865
  0.35738519 -0.15688267  0.865       0.          0.          0.
  0.        ]


In [10]:
qtorso = np.array([9.71605396e-06, 6.01925199e-05])
print(qtorso)

[9.71605396e-06 6.01925199e-05]


In [11]:
sample_joint = data_np[0, 11, 1:1+joint_dim]
sample_joint

array([ 1.31474233,  1.15303148,  1.13525614,  1.44941022, -0.63866302,
        0.93864413,  0.59152461])

In [12]:
q9 = np.concatenate((qtorso, sample_joint), axis=0)
print(q9)
print(q9.shape)

[ 9.71605396e-06  6.01925199e-05  1.31474233e+00  1.15303148e+00
  1.13525614e+00  1.44941022e+00 -6.38663018e-01  9.38644129e-01
  5.91524606e-01]
(9,)


In [13]:
tix = 1
p, R = kin.forwardkin(tix, q9)

print(p)
print(p.shape)
print(R)
print(R.shape)

[ 0.4524836  -0.16889493  1.22793331]
(3,)
[[-0.99885948  0.04715428  0.00749698]
 [ 0.04687056  0.99831029 -0.0343469 ]
 [-0.00910391 -0.03395634 -0.99938185]]
(3, 3)


In [15]:
print(R.flatten())
print(R.flatten().shape)

[-0.99885948  0.04715428  0.00749698  0.04687056  0.99831029 -0.0343469
 -0.00910391 -0.03395634 -0.99938185]
(9,)


In [16]:
# Create rotation object
rot = Rotation.from_matrix(R)

# Convert to Euler angles
euler = rot.as_euler('xyz', degrees=False)
print(euler)  # [roll, pitch, yaw] in degrees

[-3.10762838  0.00910404  3.09470297]


# Stat

In [19]:
all_eulers = []
all_pos = []
all_Rs = []

In [20]:
qtorso = np.array([9.71605396e-06, 6.01925199e-05])
print(qtorso)

[9.71605396e-06 6.01925199e-05]


In [21]:
for i_traj in range(data_np.shape[0]):
    for j_step in range(data_np.shape[1]):
        data_step = data_np[i_traj, j_step, :]
        joint_vals = data_step[1 : 1+joint_dim]
        q9 = np.concatenate((qtorso, joint_vals), axis=0)
        p, R = kin.forwardkin(tix, q9)
        euler = Rotation.from_matrix(R).as_euler('xyz', 
                                                 degrees=False)
        all_eulers.append(euler)
        all_pos.append(p)
        all_Rs.append(R.flatten())

In [22]:
all_eulers = np.array(all_eulers)
all_pos = np.array(all_pos)
all_Rs = np.array(all_Rs)

print(all_eulers.shape)
print(all_pos.shape)
print(all_Rs.shape)

(18239, 3)
(18239, 3)
(18239, 9)


In [23]:
euler_min = all_eulers.min(axis=0)
euler_max = all_eulers.max(axis=0)

print(f"Min: {euler_min}")
print(f"Max: {euler_max}")

Min: [-3.14159223 -0.06948102 -3.14159232]
Max: [3.14159251 0.19336319 3.14159256]


In [24]:
pos_min = all_pos.min(axis=0)
pos_max = all_pos.max(axis=0)

print(f"Min: {pos_min}")
print(f"Max: {pos_max}")

Min: [ 0.35035311 -0.26325649  0.86711777]
Max: [ 0.5212371  -0.07633758  1.30122593]


In [25]:
R_min = all_Rs.min(axis=0)
R_max = all_Rs.max(axis=0)

print(f"Min: {R_min}")
print(f"Max: {R_max}")

Min: [-0.99999999 -0.04946083 -0.06972123 -0.04934082  0.98773754 -0.15042152
 -0.19216048 -0.14400826 -1.        ]
Max: [-0.97770823  0.12109594  0.18069141  0.12114135  1.          0.09470949
  0.06942512  0.09212894 -0.97408447]


In [ ]:
euler_mean = all_eulers.mean(axis=0)
euler_std = all_eulers.std(axis=0)

print(f"Euler Mean: {euler_mean}")
print(f"Euler Std: {euler_std}")

# Normalize to ~N(0, 1)
def normalize_euler(euler):
    return (euler - euler_mean) / (euler_std + 1e-8)

# Position normalization (for end-effector)
pos_mean = all_pos.mean(axis=0)
pos_std = all_pos.std(axis=0)

print(f"EE Pos Mean: {pos_mean}")
print(f"EE Pos Std: {pos_std}")

def normalize_pos(pos):
    return (pos - pos_mean) / (pos_std + 1e-8)


# Rs normalization (for end-effector)
Rs_mean = all_Rs.mean(axis=0)
Rs_std = all_Rs.std(axis=0)

print(f"Rs Mean: {Rs_mean}")
print(f"Rs Std: {Rs_std}")

def normalize_Rs(Rs):
    return (Rs - Rs_mean) / (Rs_std + 1e-8)


# Target positions normalization (indices 15:24 in original data)
# 3 targets × 3 coordinates (x, y, z) = 9 values
all_target_pos = data_np[:, :, 15:24].reshape(-1, 9)
target_pos_mean = all_target_pos.mean(axis=0)
target_pos_std = all_target_pos.std(axis=0)
# target_pos_mean = np.tile(pos_mean, 3)  # [mean_x, mean_y, mean_z] × 3
# target_pos_std = np.tile(pos_std, 3)    # [std_x, std_y, std_z] × 3


print(f"Target Pos Mean: {target_pos_mean}")
print(f"Target Pos Std: {target_pos_std}")

def normalize_target_pos(target_pos):
    return (target_pos - target_pos_mean) / (target_pos_std + 1e-8)

Euler Mean: [-0.24273071  0.00634201  0.96930292]
Euler Std: [3.10951593 0.03861868 2.96679283]
EE Pos Mean: [ 0.43386127 -0.17777938  1.01862981]
EE Pos Std: [0.04539581 0.04351533 0.1066583 ]
Rs Mean: [-0.99869348  0.01723911  0.00590579  0.01607018  0.99873905 -0.00872109
 -0.00632049 -0.0081672  -0.99857101]
Rs Std: [0.00261791 0.02997263 0.03706813 0.02865225 0.00230058 0.03687495
 0.03853984 0.03539453 0.0033462 ]
Target Pos Mean: [ 0.43386127 -0.17777938  1.01862981  0.43386127 -0.17777938  1.01862981
  0.43386127 -0.17777938  1.01862981]
Target Pos Std: [0.04539581 0.04351533 0.1066583  0.04539581 0.04351533 0.1066583
 0.04539581 0.04351533 0.1066583 ]


# Create Dataset

In [27]:
# Second pass: build extended array
# Adding 12 values: current timestep (3 pos + 9 Rs)
data_extended = np.zeros((data_np.shape[0], 
                          data_np.shape[1], 35 + 12))
print(data_extended.shape)

(61, 299, 47)


In [28]:
for i_traj in range(data_np.shape[0]):
    for j_step in range(data_np.shape[1]):
        data_step = data_np[i_traj, j_step, :]
        joint_vals = data_step[1 : 1+joint_dim]
        q9 = np.concatenate((qtorso, joint_vals), axis=0)
        p, R = kin.forwardkin(tix, q9)
        # euler = Rotation.from_matrix(R).as_euler('xyz', 
        #                                          degrees=False)
        # euler_norm = normalize_euler(euler)
        # pos_norm = normalize_pos(p)
        
        # # Normalize target positions (indices 15:24)
        # target_pos = data_step[15:24]
        # target_pos_norm = normalize_target_pos(target_pos)

        # Append: original 35 + current (pos_norm 3 + euler_norm 3)
        data_extended[i_traj, j_step, :35] = data_np[i_traj, j_step, :]
        # # Overwrite target positions with normalized values
        # data_extended[i_traj, j_step, 15:24] = target_pos_norm
        # Add normalized ee pose
        data_extended[i_traj, j_step, 35:38] = p
        data_extended[i_traj, j_step, 38:] = R.flatten()

In [29]:
print(data_extended[0, 11, :])
# print(f"\nNormalized target positions (dims 15:24): {data_extended[0, 11, 15:24]}")
print(f"Current ee pose (pos + R): {data_extended[0, 11, 35:]}")

[ 1.10000000e+01  1.31474233e+00  1.15303148e+00  1.13525614e+00
  1.44941022e+00 -6.38663018e-01  9.38644129e-01  5.91524606e-01
 -1.78511221e-02 -3.92443068e-03 -8.91308725e-03  7.77143212e-03
 -3.90806527e-03 -1.89011501e-02  1.39139268e-03  4.43117333e-01
 -2.42614808e-01  8.65000000e-01  4.74497475e-01 -1.25502525e-01
  8.65000000e-01  3.57385192e-01 -1.56882667e-01  8.65000000e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -1.77693587e-02 -3.83580150e-03 -8.91295094e-03  7.48425979e-03
 -3.91306625e-03 -1.89848328e-02  1.31651868e-03  4.52483604e-01
 -1.68894934e-01  1.22793331e+00 -9.98859484e-01  4.71542806e-02
  7.49697610e-03  4.68705625e-02  9.98310293e-01 -3.43468982e-02
 -9.10391168e-03 -3.39563375e-02 -9.99381852e-01]
Current ee pose (pos + R): [ 0.4524836  -0.16889493  1.22793331 -0.99885948  0.04715428  0.00749698
  0.04687056  0.99831029 -0.0343469  -0.00910391 -0.03395634 -0.99938185]


In [30]:
print("Sample normalized ee poses (pos + R):")
for i in range(11, 21):
    print(f"  Step {i}: {data_extended[0, i, 35:]}")

Sample normalized ee poses (pos + R):
  Step 11: [ 0.4524836  -0.16889493  1.22793331 -0.99885948  0.04715428  0.00749698
  0.04687056  0.99831029 -0.0343469  -0.00910391 -0.03395634 -0.99938185]
  Step 12: [ 0.45431507 -0.1680215   1.2210318  -0.99876206  0.04876293  0.00982419
  0.04836669  0.99813732 -0.03718271 -0.01161903 -0.03666151 -0.99926019]
  Step 13: [ 0.45602985 -0.16709675  1.21409851 -0.99867639  0.05000454  0.01204221
  0.04948902  0.99797907 -0.03985743 -0.01401093 -0.03920872 -0.99913281]
  Step 14: [ 0.45763106 -0.16611958  1.20713469 -0.99860405  0.05089075  0.01414506
  0.05025183  0.99783824 -0.04235085 -0.01626975 -0.04158092 -0.99900266]
  Step 15: [ 0.45912203 -0.16508934  1.20014143 -0.99854615  0.05143455  0.01612706
  0.05067081  0.99771707 -0.0446443  -0.0183865  -0.04376223 -0.99887277]
  Step 16: [ 0.46050628 -0.16400582  1.19311963 -0.99850331  0.05165026  0.01798285
  0.050763    0.99761731 -0.04672076 -0.02035314 -0.04573797 -0.99874611]
  Step 17: [ 0

In [31]:
# Save extended data with normalized pose and targets
output_path = os.path.join(root_dir, 
                           f'data/torobo/{ds_name}/train_{ds_type}_eef_pos_R_noNorm.npy')
np.save(output_path, data_extended)
print(f"Saved to: {output_path}")

Saved to: /home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/data/torobo/trajs:72_blocks:3_triangle_v_scarce/train_interp_0.85_eef_pos_R_noNorm.npy


In [32]:
# Save all normalization stats for inference
norm_stats = {
    'euler_mean': euler_mean, 
    'euler_std': euler_std,
    'ee_pos_mean': pos_mean,
    'ee_pos_std': pos_std,
    'target_pos_mean': target_pos_mean,
    'target_pos_std': target_pos_std,
    'Rs_mean': Rs_mean,
    'Rs_std': Rs_std,
}
stats_path = os.path.join(root_dir, f'data/torobo/{ds_name}/eef_norm_stats_Rs_{ds_type}.pkl')
with open(stats_path, 'wb') as f:
    pickle.dump(norm_stats, f)
print(f"Saved normalization stats to: {stats_path}")

Saved normalization stats to: /home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/data/torobo/trajs:72_blocks:3_triangle_v_scarce/eef_norm_stats_Rs_interp_0.85.pkl


In [33]:
# Load and verify the saved stats
with open(stats_path, "rb") as f:
    data_pkl = pickle.load(f)

print("Saved normalization stats:")
for key, val in data_pkl.items():
    print(f"  {key}: {val}")

Saved normalization stats:
  euler_mean: [-0.24273071  0.00634201  0.96930292]
  euler_std: [3.10951593 0.03861868 2.96679283]
  ee_pos_mean: [ 0.43386127 -0.17777938  1.01862981]
  ee_pos_std: [0.04539581 0.04351533 0.1066583 ]
  target_pos_mean: [ 0.43386127 -0.17777938  1.01862981  0.43386127 -0.17777938  1.01862981
  0.43386127 -0.17777938  1.01862981]
  target_pos_std: [0.04539581 0.04351533 0.1066583  0.04539581 0.04351533 0.1066583
 0.04539581 0.04351533 0.1066583 ]
  Rs_mean: [-0.99869348  0.01723911  0.00590579  0.01607018  0.99873905 -0.00872109
 -0.00632049 -0.0081672  -0.99857101]
  Rs_std: [0.00261791 0.02997263 0.03706813 0.02865225 0.00230058 0.03687495
 0.03853984 0.03539453 0.0033462 ]


In [35]:
# Verify normalization - all dimensions should have similar scales
ee_repr = data_extended[:, :, 35:].reshape(-1, 12)
target_repr = data_extended[:, :, 15:24].reshape(-1, 9)

print("=== Verification: Post-normalization statistics ===")
print(f"\nEE repr (pos + Rs) - dims 35::")
print(f"  Mean: {ee_repr.mean(axis=0)}")
print(f"  Std:  {ee_repr.std(axis=0)}")

print(f"\nTarget positions - dims 15:24:")
print(f"  Mean: {target_repr.mean(axis=0)}")
print(f"  Std:  {target_repr.std(axis=0)}")

=== Verification: Post-normalization statistics ===

EE repr (pos + Rs) - dims 35::
  Mean: [ 0.43386127 -0.17777938  1.01862981 -0.99869348  0.01723911  0.00590579
  0.01607018  0.99873905 -0.00872109 -0.00632049 -0.0081672  -0.99857101]
  Std:  [0.04539581 0.04351533 0.1066583  0.00261791 0.02997263 0.03706813
 0.02865225 0.00230058 0.03687495 0.03853984 0.03539453 0.0033462 ]

Target positions - dims 15:24:
  Mean: [ 0.42621263 -0.1734421   0.865       0.42304451 -0.17472878  0.865
  0.42574286 -0.17682912  0.865     ]
  Std:  [4.89900887e-02 4.99607215e-02 1.88848936e-13 4.96197193e-02
 4.93354430e-02 1.88848936e-13 4.98197591e-02 4.91334315e-02
 1.88848936e-13]
